Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computación  
CC3084 - Data Science - sección 40

Integrantes:  
- Cristian Túnchez (231359)
- Dulce Ambrosio (231143)
- Daniel Chet (231177)
- Javier Linares (231135)

# 03. Modelado clásico y comparación de algoritmos

**MAP - Charting Student Math Misunderstandings**

En este cuaderno se construyen y comparan seis modelos de clasificación. El objetivo es determinar cuáles resuelven mejor el problema y, sobre todo, cuánto de su rendimiento proviene de leer realmente la explicación del estudiante.

El trabajo se organiza así:

1. **Preparación.** Carga del conjunto limpio y definición de las dos etapas del problema.
2. **La métrica.** Qué es MAP@3 y por qué la competencia la usa.
3. **Ajuste de hiperparámetros.** Búsqueda acotada para cada modelo.
4. **Etapa 1.** Predicción de `Category`, con los dos esquemas de validación.
5. **Etapa 2.** Predicción de `Misconception`.
6. **Objetivo compuesto.** Composición de ambas etapas y evaluación con MAP@3.
7. **Comparación y conclusiones.**

Los modelos y las métricas se guardan para que los usen el cuaderno de comparación y la aplicación.

## 0. Arranque

Este cuaderno funciona igual en **Google Colab** y en una instalación local.

En Colab la celda siguiente clona el repositorio, instala las dependencias y deja el proyecto listo. La guía de configuración en Colab está en el [README](../README.md) principal.

En local la celda no hace nada salvo ubicar la raíz del proyecto.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPOSITORIO = "https://github.com/Tunchxz/math-misconception-map.git"
CARPETA = "math-misconception-map"

if EN_COLAB:
    if not Path(CARPETA).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPOSITORIO, CARPETA], check=True)
    os.chdir(f"/content/{CARPETA}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
    )
    try:
        from google.colab import userdata

        os.environ["KAGGLE_ACCESS_TOKEN"] = userdata.get("KAGGLE_ACCESS_TOKEN")
        print("Token de Kaggle cargado desde los secretos de Colab.")
    except Exception:
        print("Registre el secreto KAGGLE_ACCESS_TOKEN para poder descargar los datos.")

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Entorno:", "Google Colab" if EN_COLAB else "local")
print("Raíz del proyecto:", RAIZ)

Token de Kaggle cargado desde los secretos de Colab.
Entorno: Google Colab
Raíz del proyecto: /content/math-misconception-map


In [2]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

import joblib

from src.config import MODELS, DATA_TRANSFORMED, SEMILLA, SIN_MISCONCEPTION
from src.data.make_dataset import asegurar_datos, guardar_transformado
from src.features import build_features as bf
from src.models import train_model as tm
from src.visualization.visualize import set_estilo

set_estilo()
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 60)

MODELS.mkdir(parents=True, exist_ok=True)
print("Semilla:", SEMILLA)

Semilla: 42


## 1. Preparación de los datos

Se parte del conjunto limpio producido en el cuaderno 01. Sobre él se aplica la unificación de
las dos grafías de la misma etiqueta que se detectó durante la limpieza.

In [3]:
train = bf.preparar_objetivos(asegurar_datos())

print("Observaciones:", f"{len(train):,}")
print("Clases de Category:", train["Category"].nunique())
print("Etiquetas de misconception tras unificar:", train["misconception_unificada"].nunique() - 1)
print("Objetivo compuesto, combinaciones:", train["objetivo_unificado"].nunique())
print()
print("Correspondencia de las grafías unificadas:")
print(bf.tabla_alias(train).to_string(index=False))

Faltan ['train.csv', 'test.csv', 'sample_submission.csv'] en /content/math-misconception-map/data/raw.
Faltan ['train.csv', 'test.csv', 'sample_submission.csv']. Descargando desde Kaggle...
  train.csv: 7,936,617 bytes
  test.csv: 687 bytes
  sample_submission.csv: 240 bytes
Reconstruyendo el conjunto limpio...
Guardado en /content/math-misconception-map/data/processed/train_limpio.parquet con forma (36696, 24)
Observaciones: 36,696
Clases de Category: 6
Etiquetas de misconception tras unificar: 34
Objetivo compuesto, combinaciones: 64

Correspondencia de las grafías unificadas:
 QuestionId       canonica       original
      31777 Wrong_fraction Wrong_Fraction
      33471 Wrong_fraction Wrong_fraction


In [4]:
COLUMNAS = ["explicacion_min", "QuestionId", "MC_Answer"] + bf.NUMERICAS

X = train[COLUMNAS]
grupos = train["QuestionId"].values

y_etapa1 = train["Category"].values

con_error = bf.filas_con_misconception(train)
X2 = con_error[COLUMNAS]
y_etapa2 = con_error["misconception_unificada"].values
grupos2 = con_error["QuestionId"].values

print(f"Etapa 1: {X.shape[0]:,} filas, {len(set(y_etapa1))} clases")
print(f"Etapa 2: {X2.shape[0]:,} filas, {len(set(y_etapa2))} clases")
print(f"Preguntas distintas: {len(set(grupos))}")

Etapa 1: 36,696 filas, 6 clases
Etapa 2: 9,860 filas, 34 clases
Preguntas distintas: 15


### Por qué dos etapas

El análisis exploratorio mostró que las dos partes del problema tienen dificultad muy distinta.
Decidir la categoría general depende poco de la pregunta concreta, mientras que identificar el
error conceptual específico depende casi por completo de ella. Separarlas permite medir cada una
con su propia métrica en lugar de promediar dos cosas distintas.

La etapa 2 se entrena únicamente con las filas que efectivamente tienen un error conceptual,
porque en las demás la pregunta no aplica.

## 2. La métrica MAP@3

La competencia admite hasta tres predicciones por observación, separadas por espacio, y las
evalúa con la precisión promedio media hasta la tercera posición. Cada observación aporta 1 si
la respuesta correcta está en primer lugar, un medio si está en segundo, un tercio si está en
tercero y cero si no aparece.

Esto reconoce que la tarea es ambigua. Antes de usarla se verifica la implementación con un
ejemplo construido a mano.

In [5]:
clases_demo = np.array(["A", "B", "C", "D"])
puntajes_demo = np.array([
    [0.1, 0.5, 0.3, 0.1],   # orden: B, C, A
    [0.6, 0.2, 0.1, 0.1],   # orden: A, B, C
    [0.1, 0.1, 0.1, 0.7],   # orden: D, A, B
])
verdad_demo = np.array(["B", "C", "X"])

print("Las tres candidatas de cada fila:", tm.formato_envio(puntajes_demo, clases_demo))
print()
print("Fila 1: 'B' en posición 1 aporta 1/1 = 1.000")
print("Fila 2: 'C' en posición 3 aporta 1/3 = 0.333")
print("Fila 3: 'X' no aparece, aporta 0.000")
print(f"Promedio esperado: {(1 + 1/3 + 0) / 3:.4f}")
print(f"Valor calculado:   {tm.map_at_k(verdad_demo, puntajes_demo, clases_demo, k=3):.4f}")

Las tres candidatas de cada fila: ['B C A', 'A B C', 'D A B']

Fila 1: 'B' en posición 1 aporta 1/1 = 1.000
Fila 2: 'C' en posición 3 aporta 1/3 = 0.333
Fila 3: 'X' no aparece, aporta 0.000
Promedio esperado: 0.4444
Valor calculado:   0.4444


## 3. Los modelos comparados

Se comparan seis modelos. Los dos primeros son líneas base que no leen el texto y existen para
poder medir, por diferencia, cuánto aporta leerlo.

In [6]:
catalogo = tm.catalogo_modelos()

resumen_modelos = pd.DataFrame([
    {
        "modelo": m.nombre,
        "usa_el_texto": "sí" if m.usa_texto else "no",
        "hiperparametros_a_ajustar": ", ".join(m.rejilla) if m.rejilla else "ninguno",
    }
    for m in catalogo
])
resumen_modelos

,modelo,usa_el_texto,hiperparametros_a_ajustar
0,Linea base (mayoritaria),no,ninguno
1,Linea base por contexto,no,ninguno
2,Regresion logistica,sí,clf__C
3,LinearSVC,sí,clf__C
4,Complement NB,sí,clf__alpha
5,Gradient boosting,sí,"clf__learning_rate, clf__max_leaf_nodes"


## 4. Ajuste de hiperparámetros de la etapa 1

Para cada modelo se prueba su rejilla sobre una única partición de retención. Se usa una sola
partición y no validación cruzada completa para acotar el tiempo de cómputo, que es la
restricción práctica de este proyecto.

In [7]:
mejores_params = {}
tablas_tuning = {}

for m in catalogo:
    if not m.rejilla:
        mejores_params[m.nombre] = {}
        continue
    params, tabla = tm.tunear(m, X, y_etapa1, grupos)
    mejores_params[m.nombre] = params
    tablas_tuning[m.nombre] = tabla
    print(f"{m.nombre:24s} -> {params}")

Regresion logistica      -> {'clf__C': 4.0}
LinearSVC                -> {'clf__C': 0.5}
Complement NB            -> {'clf__alpha': 0.1}
Gradient boosting        -> {'clf__learning_rate': 0.1, 'clf__max_leaf_nodes': 31}


In [8]:
for nombre, tabla in tablas_tuning.items():
    print(f"\n=== {nombre} ===")
    print(tabla.round(4).to_string(index=False))


=== Regresion logistica ===
 clf__C  f1_macro  f1_ponderado  segundos
    4.0    0.5138        0.6864   46.9589
    2.0    0.5128        0.6801   37.2368
    1.0    0.5043        0.6732   31.2519
    0.5    0.4845        0.6566   24.9015

=== LinearSVC ===
 clf__C  f1_macro  f1_ponderado  segundos
    0.5    0.5238        0.6936   18.9628
    2.0    0.5231        0.6939   25.3044
    1.0    0.5223        0.6966   19.8657
    0.1    0.5043        0.6812   13.4179

=== Complement NB ===
 clf__alpha  f1_macro  f1_ponderado  segundos
        0.1    0.4211        0.6055    4.8638
        0.3    0.4206        0.6016    5.1929
        1.0    0.4111        0.5919    4.3517
        3.0    0.3949        0.5803    5.7105

=== Gradient boosting ===
 clf__learning_rate  clf__max_leaf_nodes  f1_macro  f1_ponderado  segundos
                0.1                   31    0.4449        0.6362   56.7553
                0.1                   63    0.4389        0.6472   61.8862
                0.2        

## 5. Ajuste de la etapa 2

Se repite la búsqueda sobre las filas que sí tienen un error conceptual.

In [9]:
mejores_params_e2 = {}
for m in catalogo:
    if not m.rejilla:
        mejores_params_e2[m.nombre] = {}
        continue
    params, _ = tm.tunear(m, X2, y_etapa2, grupos2)
    mejores_params_e2[m.nombre] = params
    print(f"{m.nombre:24s} -> {params}")

Regresion logistica      -> {'clf__C': 4.0}
LinearSVC                -> {'clf__C': 1.0}
Complement NB            -> {'clf__alpha': 0.1}
Gradient boosting        -> {'clf__learning_rate': 0.1, 'clf__max_leaf_nodes': 31}


## 6. Validación de los modelos

Cada modelo se valida con los dos esquemas. El esquema **aleatorio** reparte las filas al azar,
de modo que las mismas preguntas aparecen en entrenamiento y en prueba. El esquema **agrupado**
deja preguntas completas fuera del entrenamiento, que es la situación real de la competencia.

Las tres etapas se miden en un solo recorrido de particiones. Cada modelo se ajusta una vez por
partición y ese ajuste se reutiliza para las tres mediciones, lo que evita repetir el
entrenamiento tres veces.

In [10]:
filas = []
for m in catalogo:
    for esquema in tm.ESQUEMAS:
        resultados = tm.evaluar_completo(
            m, train, COLUMNAS, esquema,
            mejores_params.get(m.nombre, {}), mejores_params_e2.get(m.nombre, {}),
        )
        filas.extend(resultados)
        linea = " | ".join(
            f"{r['etapa'][0]}: f1m={r['f1_macro']:.3f} map3={r['map@3']:.3f}"
            for r in resultados
        )
        print(f"{m.nombre:24s} {esquema:10s} {linea}  ({resultados[0]['segundos']:.0f}s)")

metricas = pd.DataFrame(filas)

Linea base (mayoritaria) aleatorio  1: f1m=0.096 map3=0.492 | 2: f1m=0.008 map3=0.194 | 3: f1m=0.011 map3=0.466  (1s)
Linea base (mayoritaria) agrupado   1: f1m=0.095 map3=0.491 | 2: f1m=0.004 map3=0.011 | 3: f1m=0.036 map3=0.465  (2s)
Linea base por contexto  aleatorio  1: f1m=0.409 map3=0.760 | 2: f1m=0.826 map3=0.959 | 3: f1m=0.217 map3=0.744  (10s)
Linea base por contexto  agrupado   1: f1m=0.079 map3=0.282 | 2: f1m=0.015 map3=0.134 | 3: f1m=0.017 map3=0.268  (11s)
Regresion logistica      aleatorio  1: f1m=0.537 map3=0.807 | 2: f1m=0.843 map3=0.954 | 3: f1m=0.391 map3=0.800  (286s)
Regresion logistica      agrupado   1: f1m=0.199 map3=0.489 | 2: f1m=0.003 map3=0.014 | 3: f1m=0.024 map3=0.405  (258s)
LinearSVC                aleatorio  1: f1m=0.548 map3=0.813 | 2: f1m=0.848 map3=0.958 | 3: f1m=0.039 map3=0.644  (105s)
LinearSVC                agrupado   1: f1m=0.204 map3=0.489 | 2: f1m=0.002 map3=0.013 | 3: f1m=0.060 map3=0.460  (106s)
Complement NB            aleatorio  1: f1m=0.4

### Etapa 1: predicción de `Category`

In [11]:
def vista(etapa):
    "Tabla comparativa de una etapa, con los dos esquemas lado a lado."
    sub = metricas[metricas["etapa"] == etapa]
    return (
        sub.pivot_table(index="modelo", columns="esquema",
                        values=["f1_macro", "f1_ponderado", "map@3"])
        .reindex([m.nombre for m in catalogo])
        .round(4)
    )

vista("1. Category")

f1_macro           f1_ponderado              map@3          
esquema                  agrupado aleatorio     agrupado aleatorio agrupado aleatorio
modelo                                                                               
Linea base (mayoritaria)   0.0953    0.0958       0.2320    0.2319   0.4912    0.4924
Linea base por contexto    0.0793    0.4095       0.0994    0.5996   0.2816    0.7601
Regresion logistica        0.1994    0.5367       0.2986    0.6935   0.4891    0.8069
LinearSVC                  0.2036    0.5483       0.3078    0.7040   0.4893    0.8131
Complement NB              0.2284    0.4209       0.3842    0.6108   0.5113    0.7319
Gradient boosting          0.2491    0.4564       0.4233    0.6422   0.6096    0.7832

In [12]:
def brecha(etapa):
    "Diferencia de MAP@3 entre la validacion aleatoria y la agrupada."
    sub = metricas[metricas["etapa"] == etapa]
    p = sub.pivot_table(index="modelo", columns="esquema", values="map@3")
    return (
        p.assign(
            brecha=lambda d: (d["aleatorio"] - d["agrupado"]).round(4),
            caida_pct=lambda d: ((d["aleatorio"] - d["agrupado"]) / d["aleatorio"] * 100).round(1),
        )
        .reindex([m.nombre for m in catalogo])
        .round(4)
    )

print("Caída de MAP@3 al pasar a preguntas nunca vistas, etapa 1:")
brecha("1. Category")

Caída de MAP@3 al pasar a preguntas nunca vistas, etapa 1:


esquema,agrupado,aleatorio,brecha,caida_pct
modelo,,,,
Linea base (mayoritaria),0.4912,0.4924,0.0012,0.2
Linea base por contexto,0.2816,0.7601,0.4785,63.0
Regresion logistica,0.4891,0.8069,0.3178,39.4
LinearSVC,0.4893,0.8131,0.3238,39.8
Complement NB,0.5113,0.7319,0.2205,30.1
Gradient boosting,0.6096,0.7832,0.1736,22.2


### Etapa 2: predicción de `Misconception`

Aquí el esquema agrupado es especialmente exigente. El análisis exploratorio mostró que la
mayoría de las etiquetas aparecen en una sola pregunta, de modo que al dejar esa pregunta fuera
del entrenamiento la etiqueta se vuelve imposible de predecir.

In [13]:
vista("2. Misconception")

f1_macro           f1_ponderado              map@3          
esquema                  agrupado aleatorio     agrupado aleatorio agrupado aleatorio
modelo                                                                               
Linea base (mayoritaria)   0.0043    0.0076       0.0012    0.0379   0.0113    0.1944
Linea base por contexto    0.0146    0.8257       0.1183    0.9179   0.1344    0.9593
Regresion logistica        0.0030    0.8435       0.0140    0.9258   0.0140    0.9538
LinearSVC                  0.0023    0.8480       0.0109    0.9355   0.0128    0.9577
Complement NB              0.0029    0.5995       0.0046    0.7759   0.0094    0.8715
Gradient boosting          0.0017    0.5122       0.0074    0.7380   0.0075    0.7967

In [14]:
print("Clases que el modelo nunca vio durante el entrenamiento, promedio por partición:")
print(
    metricas[metricas["etapa"] == "2. Misconception"]
    .pivot_table(index="modelo", columns="esquema", values="clases_no_vistas")
    .round(1).to_string()
)
print()
print(f"Total de etiquetas distintas en la etapa 2: {len(set(y_etapa2))}")

Clases que el modelo nunca vio durante el entrenamiento, promedio por partición:
esquema                   agrupado  aleatorio
modelo                                       
Complement NB                  6.2        0.0
Gradient boosting              6.2        0.0
Linea base (mayoritaria)       6.2        0.0
Linea base por contexto        6.2        0.0
LinearSVC                      6.2        0.0
Regresion logistica            6.2        0.0

Total de etiquetas distintas en la etapa 2: 34


### Objetivo compuesto

La competencia evalúa la etiqueta completa `Category:Misconception`. Se compone a partir de las
dos etapas multiplicando sus probabilidades: la probabilidad de una combinación es la
probabilidad de la categoría por la probabilidad del error conceptual dado que lo hay. Para las
categorías que no admiten error conceptual, la combinación termina en `NA` y conserva la
probabilidad de la categoría.

In [15]:
vista("3. Compuesto")

f1_macro           f1_ponderado              map@3          
esquema                  agrupado aleatorio     agrupado aleatorio agrupado aleatorio
modelo                                                                               
Linea base (mayoritaria)   0.0356    0.0108       0.2320    0.2319   0.4645    0.4659
Linea base por contexto    0.0172    0.2170       0.0487    0.5628   0.2681    0.7437
Regresion logistica        0.0239    0.3915       0.2396    0.6907   0.4052    0.7996
LinearSVC                  0.0604    0.0390       0.2582    0.4738   0.4597    0.6438
Complement NB              0.0380    0.2513       0.3162    0.5761   0.4356    0.6973
Gradient boosting          0.0305    0.2432       0.3503    0.6051   0.5109    0.7436

In [16]:
print("Caída de MAP@3 en el objetivo compuesto:")
brecha("3. Compuesto")

Caída de MAP@3 en el objetivo compuesto:


esquema,agrupado,aleatorio,brecha,caida_pct
modelo,,,,
Linea base (mayoritaria),0.4645,0.4659,0.0014,0.3
Linea base por contexto,0.2681,0.7437,0.4756,63.9
Regresion logistica,0.4052,0.7996,0.3944,49.3
LinearSVC,0.4597,0.6438,0.1841,28.6
Complement NB,0.4356,0.6973,0.2616,37.5
Gradient boosting,0.5109,0.7436,0.2326,31.3


## 7. Cuánto aporta leer la explicación

Esta es la comparación central del proyecto. Se contrasta el mejor modelo que lee el texto
contra la línea base que solo conoce la pregunta y la opción seleccionada.

In [17]:
base = "Linea base por contexto"
nombres_con_texto = [m.nombre for m in catalogo if m.usa_texto]

comparacion = []
for etapa in sorted(metricas["etapa"].unique()):
    for esquema in tm.ESQUEMAS:
        sub = metricas[(metricas["etapa"] == etapa) & (metricas["esquema"] == esquema)]
        con_texto = sub[sub["modelo"].isin(nombres_con_texto)].dropna(subset=["map@3"])
        linea = sub[sub["modelo"] == base]
        if con_texto.empty or linea.empty:
            continue
        mejor = con_texto.loc[con_texto["map@3"].idxmax()]
        comparacion.append({
            "etapa": etapa,
            "esquema": esquema,
            "mejor_con_texto": mejor["modelo"],
            "map3_con_texto": round(mejor["map@3"], 4),
            "map3_solo_contexto": round(linea.iloc[0]["map@3"], 4),
            "ganancia": round(mejor["map@3"] - linea.iloc[0]["map@3"], 4),
        })

comparacion = pd.DataFrame(comparacion)
comparacion

,etapa,esquema,mejor_con_texto,map3_con_texto,map3_solo_contexto,ganancia
0,1. Category,aleatorio,LinearSVC,0.8131,0.7601,0.0530
1,1. Category,agrupado,Gradient boosting,0.6096,0.2816,0.3280
2,2. Misconception,aleatorio,LinearSVC,0.9577,0.9593,-0.0017
3,2. Misconception,agrupado,Regresion logistica,0.0140,0.1344,-0.1203
4,3. Compuesto,aleatorio,Regresion logistica,0.7996,0.7437,0.0559
5,3. Compuesto,agrupado,Gradient boosting,0.5109,0.2681,0.2428


## 8. Guardado de resultados y modelos

Se guardan las métricas de todos los modelos y los artefactos de cada uno, para que los usen el
cuaderno de comparación y la aplicación.

In [18]:
guardar_transformado(metricas, "metricas_clasicos")
guardar_transformado(comparacion, "comparacion_texto_contexto")
guardar_transformado(bf.tabla_alias(train), "tabla_alias")

mejor_nombre = (
    metricas[(metricas["etapa"] == "3. Compuesto") & (metricas["esquema"] == "agrupado")]
    .sort_values("map@3", ascending=False).iloc[0]["modelo"]
)
print("Mejor modelo en el objetivo compuesto con validación agrupada:", mejor_nombre)

Mejor modelo en el objetivo compuesto con validación agrupada: Gradient boosting


In [19]:
sub_e2 = train[train["misconception_unificada"] != SIN_MISCONCEPTION]
guardados = []

for m in catalogo:
    if m.nombre == "Linea base (mayoritaria)":
        continue
    e1 = tm.entrenar_final(m, X, y_etapa1, mejores_params.get(m.nombre, {}))
    e2 = tm.entrenar_final(m, sub_e2[COLUMNAS], sub_e2["misconception_unificada"].values,
                           mejores_params_e2.get(m.nombre, {}))
    ruta = MODELS / f"{m.nombre.lower().replace(' ', '_')}.joblib"
    joblib.dump({"etapa1": e1, "etapa2": e2, "nombre": m.nombre}, ruta, compress=3)
    guardados.append({"modelo": m.nombre, "archivo": ruta.name,
                      "MB": round(ruta.stat().st_size / 1e6, 1)})

joblib.dump(
    {"columnas": COLUMNAS, "params_e1": mejores_params, "params_e2": mejores_params_e2},
    MODELS / "configuracion.joblib",
)
pd.DataFrame(guardados)

,modelo,archivo,MB
0,Linea base por contexto,linea_base_por_contexto.joblib,0.0
1,Regresion logistica,regresion_logistica.joblib,4.5
2,LinearSVC,linearsvc.joblib,3.4
3,Complement NB,complement_nb.joblib,6.1
4,Gradient boosting,gradient_boosting.joblib,47.9


### Llevar los artefactos a la máquina local

Al ejecutar en Colab, los modelos y las métricas viven en un entorno temporal. La celda
siguiente los empaqueta y los descarga para que la aplicación pueda usarlos en local.

In [20]:
import shutil

paquete = Path("artefactos_modelado")
paquete.mkdir(exist_ok=True)
shutil.copytree(MODELS, paquete / "models", dirs_exist_ok=True)
shutil.copytree(DATA_TRANSFORMED, paquete / "transformed", dirs_exist_ok=True)
shutil.make_archive("artefactos_modelado", "zip", paquete)

tamano = Path("artefactos_modelado.zip").stat().st_size / 1e6
print(f"artefactos_modelado.zip generado ({tamano:.1f} MB)")

if EN_COLAB:
    from google.colab import files

    files.download("artefactos_modelado.zip")
    print()
    print("En el repositorio local, descomprima así:")
    print("  unzip -o artefactos_modelado.zip -d /tmp/artefactos")
    print("  cp -r /tmp/artefactos/models/. models/")
    print("  cp -r /tmp/artefactos/transformed/. data/transformed/")

artefactos_modelado.zip generado (61.9 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


En el repositorio local, descomprima así:
  unzip -o artefactos_modelado.zip -d /tmp/artefactos
  cp -r /tmp/artefactos/models/. models/
  cp -r /tmp/artefactos/transformed/. data/transformed/
